In [1]:
import pandas as pd

In [2]:
def load_data(file_path):
    """Loads the stock dataset and performs initial sorting."""
    print("Loading data...")
    df = pd.read_csv(file_path)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(by=['Symbol', 'Date'])
    return df

In [10]:
def handle_missing_values(df):
    """Handles missing data using forward fill and removes empty symbols."""
    print("Handling missing values...")
    columns_to_fill = ['Close', 'Adj Close', 'High', 'Low', 'Open', 'Volume']
    
    # Forward fill within each symbol group
    for col in columns_to_fill:
        df[col] = df.groupby('Symbol')[col].ffill()
    
    # Identify symbols that have no data at all and remove them
    counts = df.groupby('Symbol')['Volume'].count()
    valid_symbols = counts[counts != 0].index.tolist()
    df = df[df['Symbol'].isin(valid_symbols)].copy()
    
    # Drop rows where 'Close' is still NaN (typically the earliest records for a stock)
    df.dropna(subset=['Close'], inplace=True)
    return df

In [11]:
def engineer_features(df):
    """Creates new features like Daily_Return, Year, and Month."""
    print("Engineering features...")
    # Calculate daily percentage change per symbol
    df['Daily_Return'] = df.groupby('Symbol')['Close'].pct_change()
    
    # Extract time-based features
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    
    # Drop the first row of each symbol (where Daily_Return is NaN)
    df.dropna(subset=['Daily_Return'], inplace=True)
    return df

In [12]:
def remove_outliers(df, columns_to_clean):
    """Removes outliers using the IQR method grouped by Symbol."""
    print("Removing outliers...")
    for column in columns_to_clean:
        # Calculate IQR for each column per symbol
        q_1 = df.groupby('Symbol')[column].transform(lambda x: x.quantile(0.25))
        q_3 = df.groupby('Symbol')[column].transform(lambda x: x.quantile(0.75))
        iqr = q_3 - q_1
        
        lower_bound = q_1 - 1.5 * iqr
        upper_bound = q_3 + 1.5 * iqr
        
        # Filter rows within bounds
        df = df[((df[column] >= lower_bound) & (df[column] <= upper_bound)) | df[column].isna()]
    
    return df

In [13]:
def run_preprocessing_pipeline(input_csv, output_csv):
    """Orchestrates the full preprocessing workflow."""
    df = load_data(input_csv)
    
    df = handle_missing_values(df)
    
    df = engineer_features(df)
    
    cols_to_clean = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Daily_Return']
    df = remove_outliers(df, cols_to_clean)
    
    df = df.drop_duplicates(subset=['Date', 'Symbol'])
    df = df.sort_values(by=['Date', 'Symbol'])
    
    print(f"Final dataset shape: {df.shape}")
    df.to_csv(output_csv, index=False)
    print(f"Cleaned data saved to: {output_csv}")
    return df

In [14]:
if __name__ == "__main__":
    # Execute the pipeline
    cleaned_stocks = run_preprocessing_pipeline('stocks.csv', 'cleaned.csv')

Loading data...


Handling missing values...
Engineering features...
Removing outliers...
Final dataset shape: (539652, 11)
Cleaned data saved to: cleaned.csv
